# Compression-01b — Kraft et McMillan en Lean 4 : le carnet compagnon du lake

Ce carnet est le **compagnon exécutable** du lake `Compression_lean` (EPIC #18706, décision D4) : il importe les modules `PrefixCodes`, `Decode` et `McMillan` **réels** du lake et les fait travailler — aucune définition n'est recopiée, ce que vous voyez exécuter est le livrable formel lui-même.

**Position dans la série** — le lake est le jumeau formel des carnets Python :
- `Compression-01-ShannonFano-Prefixe-Python.ipynb` (pli 1) : codes préfixes, Shannon–Fano, **inégalité de Kraft** → ancré par `PrefixCodes` ;
- `Compression-02-HuffmanCanonique-Python.ipynb` (pli 2, en livraison) : tables de décodage → ancré par `Decode` ;
- le **théorème de McMillan** (réciproque de Kraft) → ancré par `McMillan` (pli 3).

**Convention d'exécution** : carnet exécuté depuis la racine du lake (le kernel `lean4-wsl` détecte le `lakefile.lean` et résout les imports via `lake env`). C'est pourquoi ce fichier vit dans `Compression_lean/` : son répertoire EST son environnement.

## 0. Charger le lake

Les trois modules racine du lake : `PrefixCodes` (codes préfixes, Kraft), `Decode` (décodage, round-trip), `McMillan` (réciproque de Kraft). Le namespace commun est `CompressionLean`.

In [1]:
import PrefixCodes
import Decode
import McMillan
-- L'arc Huffman de la section 4. En Lean, un `import` doit ouvrir le fichier :
-- les dix modules se chargent donc ici, avec ceux du pli Kraft-McMillan.
import Huffman
import SymbolCodec
import Optimalite
import Appariement
import Soeurs
import Descente
import DeuxMinima
import Echange
import Expansion
import HuffmanOptimal

open CompressionLean

#check kraft_le_one
#check decode_encode
#check mcmillan_converse

import PrefixCodes
import Decode
import McMillan
-- L'arc Huffman de la section 4. En Lean, un `import` doit ouvrir le fichier :
-- les dix modules se chargent donc ici, avec ceux du pli Kraft-McMillan.
import Huffman
import SymbolCodec
import Optimalite
import Appariement
import Soeurs
import Descente
import DeuxMinima
import Echange
import Expansion
import HuffmanOptimal

open CompressionLean

#check kraft_le_one
──────▶  CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1
#check decode_encode
──────▶  CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)
  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws
#check mcmillan_converse
──────▶  CompressionLean.mcmillan_converse (L : ℕ) (ls : List ℕ) (hsort : SortedLe ls) (hL : ∀ l ∈ ls, l ≤ L)
  (hmass : mass L ls ≤ 2 ^ L) : ∃ ws, List.Pairwise PrefixFreeR ws ∧ List.map List.length ws = ls
--% env 0

Raw input:
{"cmd": "import PrefixCodes\nimport Decode\nimport McMillan\n-- L'arc Huffman de la section 4. En Lean, un `import` doit ouvrir le fichier :\n-- les dix modules se chargent donc ici, avec ceux du pli Kraft-McMillan.\nimport Huffman\nimport SymbolCodec\nimport Optimalite\nimport Appariement\nimport Soeurs\nimport Descente\nimport DeuxMinima\nimport Echange\nimport Expansion\nimport HuffmanOptimal\n\nopen CompressionLean\n\n#check kraft_le_one\n#check decode_encode\n#check mcmillan_converse"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 19, "column": 0},
   "endPos": {"line": 19, "column": 6},
   "data": "CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1"},
  {"severity": "info",
   "pos": {"line": 20, "column": 0},
   "endPos": {"line": 20, "column": 6},
   "data":
   "CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)\n  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws"},
  {"severity": "info",
   "pos": {"line": 21, "column": 0},
   "endPos": {"line": 21, "column": 6},
   "data":
   "CompressionLean.mcmillan_converse (L : ℕ) (ls : List ℕ) (hsort : SortedLe ls) (hL : ∀ l ∈ ls, l ≤ L)\n  (hmass : mass L ls ≤ 2 ^ L) : ∃ ws, List.Pairwise PrefixFreeR ws ∧ List.map List.length ws = ls"}],
 "env": 0}

Les trois résultats centraux, lus comme des contrats :

- `kraft_le_one` : la somme de Kraft de **tout** arbre de code est ≤ 1 — aucune implosion possible ;
- `decode_encode` : décoder la concaténation de mots de feuilles redonne **exactement** la liste émise — le contrat d'une table de décodage ;
- `mcmillan_converse` : si une bande de longueurs vérifie Kraft (masse ≤ 2^L), alors un code préfixe avec **exactement ces longueurs** existe — la réciproque, constructive.

## 1. Codes préfixes et somme de Kraft

Pour un arbre de code binaire `t`, `leafWords t` est la liste des mots des feuilles et `kraftSum t` la somme Σ 2^(-|w|). Le lake fournit l'arbre d'exemple `sf4` — le code Shannon–Fano à 4 symboles du carnet 01 (longueurs 1, 2, 3, 3).

In [2]:
-- Le code sf4 : ses mots de feuilles et sa somme de Kraft
#eval leafWords sf4
#eval kraftSum sf4
#eval (leafWords sf4).map List.length

-- Le code sf4 : ses mots de feuilles et sa somme de Kraft
#eval leafWords sf4
─────▶  [[true], [false, true], [false, false, true], [false, false, false]]
#eval kraftSum sf4
─────▶  1
#eval (leafWords sf4).map List.length
─────▶  [1, 2, 3, 3]
--% env 1

Raw input:
{"cmd": "-- Le code sf4 : ses mots de feuilles et sa somme de Kraft\n#eval leafWords sf4\n#eval kraftSum sf4\n#eval (leafWords sf4).map List.length", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[[true], [false, true], [false, false, true], [false, false, false]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "[1, 2, 3, 3]"}],
 "env": 1}

In [3]:
-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,
-- Kraft sature exactement a 1.
#eval kraftSum (full 3)
#eval (leafWords (full 3)).length
#check kraft_le_one
#check kraft_full

-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,
-- Kraft sature exactement a 1.
#eval kraftSum (full 3)
─────▶  1
#eval (leafWords (full 3)).length
─────▶  8
#check kraft_le_one
──────▶  CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1
#check kraft_full
──────▶  CompressionLean.kraft_full (k : ℕ) : kraftSum (full k) = 1
--% env 2

Raw input:
{"cmd": "-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,\n-- Kraft sature exactement a 1.\n#eval kraftSum (full 3)\n#eval (leafWords (full 3)).length\n#check kraft_le_one\n#check kraft_full", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "8"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "CompressionLean.kraft_full (k : ℕ) : kraftSum (full k) = 1"}],
 "env": 2}

**Lecture** — `sf4` (longueurs 1, 2, 3, 3) sature aussi Kraft à 1 : chaque bit de la bande est consommé. `full 3` est l'arbre complet à 8 feuilles : Kraft = 1 = 2^3/2^3. Le théorème `kraft_le_one` garantit qu'aucun arbre ne dépasse 1 — c'est l'argument de non-contradiction du carnet 01.

### Exercice 1 — Prédire avant d'exécuter

Un arbre complet de profondeur 5 a combien de mots de feuilles, et quelle somme de Kraft ? Écrivez vos deux prédictions sur papier, puis exécutez la cellule.

In [4]:
-- Exercice 1 : verifiez vos predictions
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval (leafWords (full 5)).length
#eval kraftSum (full 5)

-- Exercice 1 : verifiez vos predictions
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval (leafWords (full 5)).length
─────▶  32
#eval kraftSum (full 5)
─────▶  1
--% env 3

Raw input:
{"cmd": "-- Exercice 1 : verifiez vos predictions\n-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule\n#eval (leafWords (full 5)).length\n#eval kraftSum (full 5)", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "32"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "1"}],
 "env": 3}

## 2. Décodage préfixe et round-trip

`decode1` lit **un** mot le long de l'arbre ; `decode` itère sur le flux complet. Les théorèmes `decode1_sound` / `decode1_complete` établissent la correction du pas, et `decode_encode` ferme le contrat : émettre la concaténation des mots puis décoder redonne la liste initiale, dans l'ordre, sans reste.

Le message du carnet 01 : `a b c` se code `0·10·110` (ici `[true, false, true, false, false, true]` en convention bit=Bool).

In [5]:
-- Decoder le message a b c du carnet 01
#eval decode sf4 [true, false, true, false, false, true]
-- et le message complet a b c d
#eval decode sf4 [true, false, true, false, false, true, false, false, false]

-- Decoder le message a b c du carnet 01
#eval decode sf4 [true, false, true, false, false, true]
─────▶  some [[true], [false, true], [false, false, true]]
-- et le message complet a b c d
#eval decode sf4 [true, false, true, false, false, true, false, false, false]
─────▶  some [[true], [false, true], [false, false, true], [false, false, false]]
--% env 4

Raw input:
{"cmd": "-- Decoder le message a b c du carnet 01\n#eval decode sf4 [true, false, true, false, false, true]\n-- et le message complet a b c d\n#eval decode sf4 [true, false, true, false, false, true, false, false, false]", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "some [[true], [false, true], [false, false, true]]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data":
   "some [[true], [false, true], [false, false, true], [false, false, false]]"}],
 "env": 4}

In [6]:
-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder
#eval decode sf4 (leafWords sf4).flatten
#check decode_encode

-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder
#eval decode sf4 (leafWords sf4).flatten
─────▶  some [[true], [false, true], [false, false, true], [false, false, false]]
#check decode_encode
──────▶  CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)
  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws
--% env 5

Raw input:
{"cmd": "-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder\n#eval decode sf4 (leafWords sf4).flatten\n#check decode_encode", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "some [[true], [false, true], [false, false, true], [false, false, false]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)\n  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws"}],
 "env": 5}

**Lecture** — le round-trip rend `some (leafWords sf4)` : la table est complète, l'ordre est préservé.

### Exercice 2 — Un flux tronqué

Le flux `[false]` ne contient aucun mot complet (le seul mot de longueur 1 est `a` = `[true]`). Que répond `decode` ? Prédisez, puis exécutez. Que répond-il pour `[false, true]` ?

In [7]:
-- Exercice 2 : flux tronque puis complet
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval decode sf4 [false]
#eval decode sf4 [false, true]

-- Exercice 2 : flux tronque puis complet
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval decode sf4 [false]
─────▶  none
#eval decode sf4 [false, true]
─────▶  some [[false, true]]
--% env 6

Raw input:
{"cmd": "-- Exercice 2 : flux tronque puis complet\n-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule\n#eval decode sf4 [false]\n#eval decode sf4 [false, true]", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "none"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "some [[false, true]]"}],
 "env": 6}

## 3. McMillan : la réciproque de Kraft, constructivement

Kraft dit « tout code préfixe satisfait Σ 2^(-ℓᵢ) ≤ 1 ». McMillan renverse l'énoncé : **toute** bande de longueurs triée satisfaisant l'inégalité est réalisée par un code préfixe. La preuve du lake est constructive : `canon ls c` construit le code canonique (Schwartz-Kallick 1964) par sauts de masse, `mass` mesure la bande à l'échelle 2^L (entiers, pas de rationnels), et `canon_spec` établit les deux moitiés : préfixe-liberté **et** longueurs exactes.

**Convention de bit** — `canon` démarre au bit `false` (`a = [false]`), là où le `sf4` du carnet 01 écrit `a = [true]` : les deux tables sont complémentaires bit à bit, de mêmes longueurs, et ce que Schwartz-Kallick canonise est la bande de longueurs, pas la valeur des bits.

In [8]:
-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee
#eval canon [1, 2, 3, 3] 0
#eval mass 3 [1, 2, 3, 3]
-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111

-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee
#eval canon [1, 2, 3, 3] 0
─────▶  [[false], [true, false], [true, true, false], [true, true, true]]
#eval mass 3 [1, 2, 3, 3]
─────▶  8
-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111
--% env 7

Raw input:
{"cmd": "-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee\n#eval canon [1, 2, 3, 3] 0\n#eval mass 3 [1, 2, 3, 3]\n-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[[false], [true, false], [true, true, false], [true, true, true]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "8"}],
 "env": 7}

In [9]:
-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou
#eval canon [1, 3, 3] 0
#eval mass 3 [1, 3, 3]
-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer
#eval mass 3 [1, 1, 1]

-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou
#eval canon [1, 3, 3] 0
─────▶  [[false], [true, false, false], [true, false, true]]
#eval mass 3 [1, 3, 3]
─────▶  6
-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer
#eval mass 3 [1, 1, 1]
─────▶  12
--% env 8

Raw input:
{"cmd": "-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou\n#eval canon [1, 3, 3] 0\n#eval mass 3 [1, 3, 3]\n-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer\n#eval mass 3 [1, 1, 1]", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[[false], [true, false, false], [true, false, true]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "12"}],
 "env": 8}

In [10]:
-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms mcmillan_converse
#print axioms canon_spec
#print axioms decode_encode
#print axioms kraft_le_one

-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms mcmillan_converse
──────▶  'CompressionLean.mcmillan_converse' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms canon_spec
──────▶  'CompressionLean.canon_spec' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms decode_encode
──────▶  'CompressionLean.decode_encode' depends on axioms: [propext, Quot.sound]
#print axioms kraft_le_one
──────▶  'CompressionLean.kraft_le_one' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 9

Raw input:
{"cmd": "-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native\n#print axioms mcmillan_converse\n#print axioms canon_spec\n#print axioms decode_encode\n#print axioms kraft_le_one", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'CompressionLean.mcmillan_converse' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'CompressionLean.canon_spec' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'CompressionLean.decode_encode' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'CompressionLean.kraft_le_one' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 9}

**Lecture** — les axiomes, pilier par pilier, tels que les imprime la cellule précédente : `mcmillan_converse`, `canon_spec` et `kraft_le_one` dépendent de `propext`, `Classical.choice` et `Quot.sound` ; `decode_encode` ne dépend que de `propext` et `Quot.sound` — c'est le seul des quatre qui n'utilise pas le choix classique. Aucune preuve ne contient `sorryAx`, et aucune ne passe par le noyau natif (`native_decide`). Le livrable formel est complet.

### Encart Mathlib — ce que l'organe fournit, ce que le lake ajoute

Mathlib porte déjà l'inégalité, au pin `db584cd6` (Mathlib 4.33) : module `Mathlib.InformationTheory.Coding.KraftMcMillan`, théorème `InformationTheory.kraft_mcmillan_inequality` — pour un code **uniquement décodable** `S` sur un alphabet de taille `D`, `∑ w ∈ S, (1 / D) ^ w.length ≤ 1` sur `ℝ`. L'organe existe donc : le lake ne le réimplante pas.

Ce que le lake ajoute est la **direction constructive** : `mcmillan_converse` établit que toute bande de longueurs triée, majorée par `L` et de masse `≤ 2^L` — l'hypothèse de Kraft `Σ 2^(-ℓᵢ) ≤ 1` multipliée par `2^L` — est **réalisable** par un code préfixe de ces longueurs exactes, et `canon_spec` que la construction canonique `canon` en produit un. S'y ajoute une formalisation sur `CodeTree` (`kraft_le_one`), sur les rationnels exacts, qui sert au décodage de la série. Au pin, `Mathlib/` ne contient aucune occurrence de `PrefixFree` ni de `Huffman` (recherche sur l'arbre complet) : l'encart est un **complément déclaré** à l'organe, pas une absence.

### Exercice 3 — Réaliser une bande libre

Choisissez une bande de longueurs triée, de masse **strictement inférieure** à 2^L (par exemple `[1, 3]` avec L = 3). Vérifiez : (a) sa masse, (b) que `canon` la réalise avec des longueurs exactes. Que remarquez-vous pour les mots produits ?

In [11]:
-- Exercice 3 : bande libre (1, 3)
-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez
#eval mass 3 [1, 3]
#eval canon [1, 3] 0
#eval (canon [1, 3] 0).map List.length

-- Exercice 3 : bande libre (1, 3)
-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez
#eval mass 3 [1, 3]
─────▶  5
#eval canon [1, 3] 0
─────▶  [[false], [true, false, false]]
#eval (canon [1, 3] 0).map List.length
─────▶  [1, 3]
--% env 10

Raw input:
{"cmd": "-- Exercice 3 : bande libre (1, 3)\n-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez\n#eval mass 3 [1, 3]\n#eval canon [1, 3] 0\n#eval (canon [1, 3] 0).map List.length", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "5"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "[[false], [true, false, false]]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[1, 3]"}],
 "env": 10}

## 4. Annexe — l'arc Huffman du lake : construire, payer, réarranger, conclure

Les sections 0 à 3 ont exécuté les modules du pli Kraft–McMillan (`PrefixCodes`, `Decode`, `McMillan`). Le lake ne s'arrête pas là : un second arc formalise la **construction de Huffman et la preuve de son optimalité**, module par module (vague *Compression* de [#11703](https://github.com/jsboige/CoursIA/issues/11703) — les modules restés invisibles au rejeu du 31/10) :

| Étape | Modules | Rôle |
|---|---|---|
| Construire | `Huffman`, `SymbolCodec` | forêt triée par poids, fusions successives ; passerelle symboles → fréquences |
| Payer | `Optimalite`, `Appariement` | coût d'un arbre pondéré ; lecture poids × profondeurs (inégalité de réarrangement) |
| Réarranger | `Soeurs`, `Descente`, `DeuxMinima`, `Echange` | chemins sœurs, descente des minima au fond, monotonie des échanges |
| Fusionner | `Expansion` | la loi de la fusion : remplacer une feuille `(a+b)` par un nœud facture exactement `a+b` |
| Conclure | `HuffmanOptimal` | le théorème final : l'arbre de Huffman est de coût minimal |

Cette annexe exécute l'arc sur la bande du carnet 02 : fréquences `(4, 2, 1, 1)`.

In [12]:
-- Les dix modules de l'arc, charges par la cellule d'ouverture (un `import`
-- doit ouvrir le fichier : il ne peut pas venir au milieu du carnet).
-- Un point d'entree par module, au niveau des types : c'est le contrat que
-- chaque etape du tableau ci-dessus apporte au lake.
#check @huffman_canonical      -- Huffman       : la construction
#check @toWTree                -- SymbolCodec   : la passerelle symboles -> poids
#check @cost_node_charge       -- Optimalite    : le cout, noeud par noeud
#check @cost_dot               -- Appariement   : poids x profondeurs
#check @sisterPath_spec        -- Soeurs        : le chemin des soeurs
#check @pathLen_le_maxDepth    -- Descente      : nulle part ou cacher une feuille
#check @both_mins_to_bottom    -- DeuxMinima    : les deux plus petits au fond
#check @exchange_le            -- Echange       : remonter a, descendre b
#check @costW_subst            -- Expansion     : la fusion facture (a+b)
#check @huffman_optimal        -- HuffmanOptimal: le theoreme final

-- Les dix modules de l'arc, charges par la cellule d'ouverture (un `import`
-- doit ouvrir le fichier : il ne peut pas venir au milieu du carnet).
-- Un point d'entree par module, au niveau des types : c'est le contrat que
-- chaque etape du tableau ci-dessus apporte au lake.
#check @huffman_canonical      -- Huffman       : la construction
──────▶  huffman_canonical : ∀ (ws : List ℕ),
  ∃ ws', List.Pairwise PrefixFreeR ws' ∧ List.map List.length ws' = isort (depthsW (huffman ws))
#check @toWTree                -- SymbolCodec   : la passerelle symboles -> poids
──────▶  @toWTree : {n : ℕ} → (Fin n → ℕ) → LTree n → WTree
#check @cost_node_charge       -- Optimalite    : le cout, noeud par noeud
──────▶  cost_node_charge : ∀ (l r : WTree), costW (l.node r) = costW l + costW r + weight (l.node r)
#check @cost_dot               -- Appariement   : poids x profondeurs
──────▶  cost_dot : ∀ (t : WTree), costW t = dotp (leafWeights t) (depthsW t)
#check @sisterPath_spec        -- Soeurs        : le chemin des soeurs
──────▶  sisterPath_spec : ∀ (t : WTree),
  (∃ l r, t = l.node r) → ∃ π a b, subtreeAt t π = some ((WTree.leaf a).node (WTree.leaf b)) ∧ π.length + 1 = maxDepth t
#check @pathLen_le_maxDepth    -- Descente      : nulle part ou cacher une feuille
──────▶  pathLen_le_maxDepth : ∀ (t : WTree) (π : List Bool) (s : WTree), subtreeAt t π = some s → π.length ≤ maxDepth t
#check @both_mins_to_bottom    -- DeuxMinima    : les deux plus petits au fond
──────▶  both_mins_to_bottom : ∀ (t : WTree),
  (∃ l r, t = l.node r) →
    ∀ (w₁ w₂ : ℕ),
      (∀ v ∈ leafWeights t, w₁ ≤ v) →
        w₁ ∈ leafWeights t →
          (∀ v ∈ (leafWeights t).erase w₁, w₂ ≤ v) →
            w₂ ∈ (leafWeights t).erase w₁ →
              ∃ t' π,
                (leafWeights t').Perm (leafWeights t) ∧
                  costW t' ≤ costW t ∧
                    subtreeAt t' π = some ((WTree.leaf w₁).node (WTree.leaf w₂)) ∧ π.length + 1 = maxDepth t'
#check @exchange_le            -- Echange       : remonter a, descendre b
──────▶  exchange_le : ∀ (k a b : ℕ) (r : WTree),
  a ≤ b →
    costAt k ((WTree.leaf b).node ((WTree.leaf a).node r)) ≤ costAt k ((WTree.leaf a).node ((WTree.leaf b).node r))
#check @costW_subst            -- Expansion     : la fusion facture (a+b)
──────▶  costW_subst : ∀ (t : WTree) (a b : ℕ), a + b ∈ leafWeights t → costW (subst t a b) = costW t + (a + b)
#check @huffman_optimal        -- HuffmanOptimal: le theoreme final
──────▶  huffman_optimal : ∀ (n : ℕ) (ws : List ℕ),
  ws.length ≤ n → ∀ (t : WTree), (leafWeights t).Perm ws → costW (huffman ws) ≤ costW t
--% env 11

Raw input:
{"cmd": "-- Les dix modules de l'arc, charges par la cellule d'ouverture (un `import`\n-- doit ouvrir le fichier : il ne peut pas venir au milieu du carnet).\n-- Un point d'entree par module, au niveau des types : c'est le contrat que\n-- chaque etape du tableau ci-dessus apporte au lake.\n#check @huffman_canonical      -- Huffman       : la construction\n#check @toWTree                -- SymbolCodec   : la passerelle symboles -> poids\n#check @cost_node_charge       -- Optimalite    : le cout, noeud par noeud\n#check @cost_dot               -- Appariement   : poids x profondeurs\n#check @sisterPath_spec        -- Soeurs        : le chemin des soeurs\n#check @pathLen_le_maxDepth    -- Descente      : nulle part ou cacher une feuille\n#check @both_mins_to_bottom    -- DeuxMinima    : les deux plus petits au fond\n#check @exchange_le            -- Echange       : remonter a, descendre b\n#check @costW_subst            -- Expansion     : la fusion facture (a+b)\n#check @huffman_optimal        -- HuffmanOptimal: le theoreme final", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "huffman_canonical : ∀ (ws : List ℕ),\n  ∃ ws', List.Pairwise PrefixFreeR ws' ∧ List.map List.length ws' = isort (depthsW (huffman ws))"},
  {"severity": "info",
   "pos": {"line": 

In [13]:
-- La bande (4, 2, 1, 1) : l'arbre de Huffman et son cout de fusion
#eval huffman [4, 2, 1, 1]
#eval costW (huffman [4, 2, 1, 1])

-- L'autre lecture du meme cout : poids x profondeurs, feuille par feuille
-- (cost_dot : costW t = dotp (leafWeights t) (depthsW t))
#eval leafWeights (huffman [4, 2, 1, 1])
#eval depthsW (huffman [4, 2, 1, 1])
#eval dotp (leafWeights (huffman [4, 2, 1, 1])) (depthsW (huffman [4, 2, 1, 1]))

-- Le meme arbre, relu comme arbre de code : ses mots de feuilles forment un
-- code prefixe (trie_prefixFree) -- le lien avec la partie Kraft du carnet.
#eval leafWords (toCodeTree (huffman [4, 2, 1, 1]))
#check @trie_prefixFree

-- La bande (4, 2, 1, 1) : l'arbre de Huffman et son cout de fusion
#eval huffman [4, 2, 1, 1]
─────▶  CompressionLean.WTree.node
  (CompressionLean.WTree.node
    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1))
    (CompressionLean.WTree.leaf 2))
  (CompressionLean.WTree.leaf 4)
#eval costW (huffman [4, 2, 1, 1])
─────▶  14

-- L'autre lecture du meme cout : poids x profondeurs, feuille par feuille
-- (cost_dot : costW t = dotp (leafWeights t) (depthsW t))
#eval leafWeights (huffman [4, 2, 1, 1])
─────▶  [1, 1, 2, 4]
#eval depthsW (huffman [4, 2, 1, 1])
─────▶  [3, 3, 2, 1]
#eval dotp (leafWeights (huffman [4, 2, 1, 1])) (depthsW (huffman [4, 2, 1, 1]))
─────▶  14

-- Le meme arbre, relu comme arbre de code : ses mots de feuilles forment un
-- code prefixe (trie_prefixFree) -- le lien avec la partie Kraft du carnet.
#eval leafWords (toCodeTree (huffman [4, 2, 1, 1]))
─────▶  [[true, true, true], [true, true, false], [true, false], [false]]
#check @trie_prefixFree
──────▶  trie_prefixFree : ∀ (t : WTree), IsPrefixFree (leafWords (toCodeTree t))
--% env 12

Raw input:
{"cmd": "-- La bande (4, 2, 1, 1) : l'arbre de Huffman et son cout de fusion\n#eval huffman [4, 2, 1, 1]\n#eval costW (huffman [4, 2, 1, 1])\n\n-- L'autre lecture du meme cout : poids x profondeurs, feuille par feuille\n-- (cost_dot : costW t = dotp (leafWeights t) (depthsW t))\n#eval leafWeights (huffman [4, 2, 1, 1])\n#eval depthsW (huffman [4, 2, 1, 1])\n#eval dotp (leafWeights (huffman [4, 2, 1, 1])) (depthsW (huffman [4, 2, 1, 1]))\n\n-- Le meme arbre, relu comme arbre de code : ses mots de feuilles forment un\n-- code prefixe (trie_prefixFree) -- le lien avec la partie Kraft du carnet.\n#eval leafWords (toCodeTree (huffman [4, 2, 1, 1]))\n#check @trie_prefixFree", "env": 11}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "CompressionLean.WTree.node\n  (CompressionLean.WTree.node\n    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1))\n    (CompressionLean.WTree.leaf 2))\n  (CompressionLean.WTree.leaf 4)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "14"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "[1, 1, 2, 4]"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "[3, 3, 2, 1]"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "14"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "[[true, true, true], [true, true, false], [true, false], [false]]"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 6},
   "data":
   "trie_prefixFree : ∀ (t : WTree), IsPrefixFree (leafWords (toCodeTree t))"}],
 "env": 12}

**Lecture** — les fusions se lisent dans le coût : `1+1 = 2`, puis `2+2 = 4`, puis `4+4 = 8`, pour un coût total de `14`. La lecture `dotp` donne la même valeur, feuille par feuille : `4×1 + 2×2 + 1×3 + 1×3 = 14` — c'est le théorème `cost_dot` d'`Appariement`, et l'inégalité de réarrangement `rearrange_two` du même module est ce qui force les poids lourds vers les profondeurs faibles : un appariement croissant ne peut pas coûter moins. Les modules `Soeurs`, `Descente`, `DeuxMinima` et `Echange` outillent ces déplacements locaux (chemins sœurs, envoi des deux minima au fond, monotonie d'un échange) : ce sont les seuls mouvements qu'un arbre optimal doit tolérer.

In [14]:
-- Les mouvements locaux, executes sur l'arbre de (4, 2, 1, 1).
-- Soeurs : sisterPath descend vers la paire de feuilles soeurs la plus
-- profonde -- les deux poids 1, fusionnes en premier par la construction.
#eval sisterPath (huffman [4, 2, 1, 1])
#check @sisterPath_spec

-- Descente : un sous-arbre atteint par un chemin de longueur k est a au
-- moins k de la profondeur restante -- nulle part ou cacher une feuille.
#check @pathLen_le_maxDepth

-- DeuxMinima : les deux plus petits poids peuvent toujours devenir soeurs au
-- fond, sans toucher la bande des poids de feuilles.
#check @both_mins_to_bottom

-- Echange : quand a <= b, remonter a d'un etage et descendre b ne peut pas
-- augmenter le cout -- la monotonie qui fait descendre les minima.
#check @exchange_le

-- Les mouvements locaux, executes sur l'arbre de (4, 2, 1, 1).
-- Soeurs : sisterPath descend vers la paire de feuilles soeurs la plus
-- profonde -- les deux poids 1, fusionnes en premier par la construction.
#eval sisterPath (huffman [4, 2, 1, 1])
─────▶  [true, true]
#check @sisterPath_spec
──────▶  sisterPath_spec : ∀ (t : WTree),
  (∃ l r, t = l.node r) → ∃ π a b, subtreeAt t π = some ((WTree.leaf a).node (WTree.leaf b)) ∧ π.length + 1 = maxDepth t

-- Descente : un sous-arbre atteint par un chemin de longueur k est a au
-- moins k de la profondeur restante -- nulle part ou cacher une feuille.
#check @pathLen_le_maxDepth
──────▶  pathLen_le_maxDepth : ∀ (t : WTree) (π : List Bool) (s : WTree), subtreeAt t π = some s → π.length ≤ maxDepth t

-- DeuxMinima : les deux plus petits poids peuvent toujours devenir soeurs au
-- fond, sans toucher la bande des poids de feuilles.
#check @both_mins_to_bottom
──────▶  both_mins_to_bottom : ∀ (t : WTree),
  (∃ l r, t = l.node r) →
    ∀ (w₁ w₂ : ℕ),
      (∀ v ∈ leafWeights t, w₁ ≤ v) →
        w₁ ∈ leafWeights t →
          (∀ v ∈ (leafWeights t).erase w₁, w₂ ≤ v) →
            w₂ ∈ (leafWeights t).erase w₁ →
              ∃ t' π,
                (leafWeights t').Perm (leafWeights t) ∧
                  costW t' ≤ costW t ∧
                    subtreeAt t' π = some ((WTree.leaf w₁).node (WTree.leaf w₂)) ∧ π.length + 1 = maxDepth t'

-- Echange : quand a <= b, remonter a d'un etage et descendre b ne peut pas
-- augmenter le cout -- la monotonie qui fait descendre les minima.
#check @exchange_le
──────▶  exchange_le : ∀ (k a b : ℕ) (r : WTree),
  a ≤ b →
    costAt k ((WTree.leaf b).node ((WTree.leaf a).node r)) ≤ costAt k ((WTree.leaf a).node ((WTree.leaf b).node r))
--% env 13

Raw input:
{"cmd": "-- Les mouvements locaux, executes sur l'arbre de (4, 2, 1, 1).\n-- Soeurs : sisterPath descend vers la paire de feuilles soeurs la plus\n-- profonde -- les deux poids 1, fusionnes en premier par la construction.\n#eval sisterPath (huffman [4, 2, 1, 1])\n#check @sisterPath_spec\n\n-- Descente : un sous-arbre atteint par un chemin de longueur k est a au\n-- moins k de la profondeur restante -- nulle part ou cacher une feuille.\n#check @pathLen_le_maxDepth\n\n-- DeuxMinima : les deux plus petits poids peuvent toujours devenir soeurs au\n-- fond, sans toucher la bande des poids de feuilles.\n#check @both_mins_to_bottom\n\n-- Echange : quand a <= b, remonter a d'un etage et descendre b ne peut pas\n-- augmenter le cout -- la monotonie qui fait descendre les minima.\n#check @exchange_le", "env": 12}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "[true, true]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "sisterPath_spec : ∀ (t : WTree),\n  (∃ l r, t = l.node r) → ∃ π a b, subtreeAt t π = some ((WTree.leaf a).node (WTree.leaf b)) ∧ π.length + 1 = maxDepth t"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "pathLen_le_maxDepth : ∀ (t : WTree) (π : List Bool) (s : WTree), subtreeAt t π = some s → π.length ≤ maxDepth t"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 6},
   "data":
   "both_mins_to_bottom : ∀ (t : WTree),\n  (∃ l r, t = l.node r) →\n    ∀ (w₁ w₂ : ℕ),\n      (∀ v ∈ leafWeights t, w₁ ≤ v) →\n        w₁ ∈ leafWeights t →\n          (∀ v ∈ (leafWeights t).erase w₁, w₂ ≤ v) →\n            w₂ ∈ (leafWeights t).erase w₁ →\n              ∃ t' π,\n                (leafWeights t').Perm (leafWeights t) ∧\n                  costW t' ≤ costW t ∧\n                    subtreeAt t' π = some ((WTree.leaf w₁).node (WTree.leaf w₂)) ∧ π.length + 1 = maxDepth t'"},
  {"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 6},
   "data":
   "exchange_le : ∀ (k a b : ℕ) (r :

**Lecture** — `sisterPath` désigne la paire sœur la plus profonde de l'arbre : les deux feuilles de poids `1`, fusionnées en premier par la construction. Les trois `#check` disent pourquoi un arbre mal apparié ne peut pas gagner : une feuille ne se cache pas plus profonde que l'arbre (`pathLen_le_maxDepth`), les deux minima peuvent toujours être rendus sœurs au fond (`both_mins_to_bottom`), et l'échange qui remonte le plus léger d'un étage ne coûte jamais plus (`exchange_le`). Ce sont exactement les mouvements enchaînés par la preuve d'`huffman_optimal`.

In [15]:
-- Expansion.subst : la fusion inverse -- remplacer la feuille (a+b) par le
-- noeud (a, b) facture exactement (a+b) (costW_subst)
#eval subst (huffman [4, 2, 1, 1]) 1 1
#eval costW (subst (huffman [4, 2, 1, 1]) 1 1)   -- attendu : 14 + (1+1) = 16

-- huffman_expand : la loi de recursion de la construction -- fusionner les deux
-- plus petits de la bande triee, payer exactement leur somme
#eval costW (huffman [1, 1, 2, 4])
#eval costW (huffman ([2, 4] ++ [1 + 1])) + (1 + 1)   -- la meme valeur : 14

-- Expansion.subst : la fusion inverse -- remplacer la feuille (a+b) par le
-- noeud (a, b) facture exactement (a+b) (costW_subst)
#eval subst (huffman [4, 2, 1, 1]) 1 1
─────▶  CompressionLean.WTree.node
  (CompressionLean.WTree.node
    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1))
    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1)))
  (CompressionLean.WTree.leaf 4)
#eval costW (subst (huffman [4, 2, 1, 1]) 1 1)   -- attendu : 14 + (1+1) = 16
─────▶  16

-- huffman_expand : la loi de recursion de la construction -- fusionner les deux
-- plus petits de la bande triee, payer exactement leur somme
#eval costW (huffman [1, 1, 2, 4])
─────▶  14
#eval costW (huffman ([2, 4] ++ [1 + 1])) + (1 + 1)   -- la meme valeur : 14
─────▶  14
--% env 14

Raw input:
{"cmd": "-- Expansion.subst : la fusion inverse -- remplacer la feuille (a+b) par le\n-- noeud (a, b) facture exactement (a+b) (costW_subst)\n#eval subst (huffman [4, 2, 1, 1]) 1 1\n#eval costW (subst (huffman [4, 2, 1, 1]) 1 1)   -- attendu : 14 + (1+1) = 16\n\n-- huffman_expand : la loi de recursion de la construction -- fusionner les deux\n-- plus petits de la bande triee, payer exactement leur somme\n#eval costW (huffman [1, 1, 2, 4])\n#eval costW (huffman ([2, 4] ++ [1 + 1])) + (1 + 1)   -- la meme valeur : 14", "env": 13}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data":
   "CompressionLean.WTree.node\n  (CompressionLean.WTree.node\n    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1))\n    (CompressionLean.WTree.node (CompressionLean.WTree.leaf 1) (CompressionLean.WTree.leaf 1)))\n  (CompressionLean.WTree.leaf 4)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "16"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "14"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "14"}],
 "env": 14}

**Lecture** — `subst` rouvre la première fusion : le coût passe de `14` à `16`, exactement de `1+1`. `huffman_expand` dit que la construction entière obéit à cette loi : le coût de `huffman ws` est celui de `huffman (rest ++ [w₁+w₂])` plus `w₁+w₂` — chaque fusion paie le poids du nœud qu'elle crée (`cost_node_charge` d'`Optimalite`). La chaîne fusion + réarrangement + induction aboutit au théorème central d'`HuffmanOptimal` :

> `huffman_optimal : ∀ n ws, ws.length ≤ n → ∀ t, List.Perm (leafWeights t) ws → costW (huffman ws) ≤ costW t`

— parmi **tous** les arbres de mêmes poids de feuilles, celui de la construction est de coût minimal.

In [16]:
-- SymbolCodec : la passerelle entre le trie canonique du carnet 02 (arbre
-- etiquete par symboles, LTree n, frequences Fin n -> Nat) et le WTree pondere.
-- Le fichier declare tout dans `namespace CompressionLean` (pas de sous-espace
-- `SymbolCodec`) : les noms s'ecrivent donc nus, une fois le namespace ouvert.
#check @toWTree
#check @wordsOf

-- Les trois theoremes de la passerelle, au niveau des types : profondeurs
-- (depthsW_toWTree), poids (weight_toWTree) et mots (wordsOf_forget) se
-- transportent du trie canonique vers l'arbre pondere du lake.
#check @depthsW_toWTree
#check @weight_toWTree
#check @wordsOf_forget

-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms huffman_optimal
#print axioms huffman_expand
#print axioms cost_dot
#print axioms costW_subst
#print axioms wordsOf_forget

-- SymbolCodec : la passerelle entre le trie canonique du carnet 02 (arbre
-- etiquete par symboles, LTree n, frequences Fin n -> Nat) et le WTree pondere.
-- Le fichier declare tout dans `namespace CompressionLean` (pas de sous-espace
-- `SymbolCodec`) : les noms s'ecrivent donc nus, une fois le namespace ouvert.
#check @toWTree
──────▶  @toWTree : {n : ℕ} → (Fin n → ℕ) → LTree n → WTree
#check @wordsOf
──────▶  @wordsOf : {n : ℕ} → LTree n → List (List Bool)

-- Les trois theoremes de la passerelle, au niveau des types : profondeurs
-- (depthsW_toWTree), poids (weight_toWTree) et mots (wordsOf_forget) se
-- transportent du trie canonique vers l'arbre pondere du lake.
#check @depthsW_toWTree
──────▶  @depthsW_toWTree : ∀ {n : ℕ} (t : LTree n) (wt : Fin n → ℕ),
  depthsW (toWTree wt t) = List.map (fun p => p.1.length) (pairs t)
#check @weight_toWTree
──────▶  @weight_toWTree : ∀ {n : ℕ} (t : LTree n) (wt : Fin n → ℕ), weight (toWTree wt t) = (List.map wt (labels t)).sum
#check @wordsOf_forget
──────▶  @wordsOf_forget : ∀ {n : ℕ} (t : LTree n), wordsOf t = leafWords (forget t)

-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms huffman_optimal
──────▶  'CompressionLean.huffman_optimal' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms huffman_expand
──────▶  'CompressionLean.huffman_expand' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms cost_dot
──────▶  'CompressionLean.cost_dot' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms costW_subst
──────▶  'CompressionLean.costW_subst' depends on axioms: [propext, Quot.sound]
#print axioms wordsOf_forget
──────▶  'CompressionLean.wordsOf_forget' depends on axioms: [propext]
--% env 15

Raw input:
{"cmd": "-- SymbolCodec : la passerelle entre le trie canonique du carnet 02 (arbre\n-- etiquete par symboles, LTree n, frequences Fin n -> Nat) et le WTree pondere.\n-- Le fichier declare tout dans `namespace CompressionLean` (pas de sous-espace\n-- `SymbolCodec`) : les noms s'ecrivent donc nus, une fois le namespace ouvert.\n#check @toWTree\n#check @wordsOf\n\n-- Les trois theoremes de la passerelle, au niveau des types : profondeurs\n-- (depthsW_toWTree), poids (weight_toWTree) et mots (wordsOf_forget) se\n-- transportent du trie canonique vers l'arbre pondere du lake.\n#check @depthsW_toWTree\n#check @weight_toWTree\n#check @wordsOf_forget\n\n-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native\n#print axioms huffman_optimal\n#print axioms huffman_expand\n#print axioms cost_dot\n#print axioms costW_subst\n#print axioms wordsOf_forget", "env": 14}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "@toWTree : {n : ℕ} → (Fin n → ℕ) → LTree n → WTree"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "@wordsOf : {n : ℕ} → LTree n → List (List Bool)"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "@depthsW_toWTree : ∀ {n : ℕ} (t : LTree n) (wt : Fin n → ℕ),\n  depthsW (toWTree wt t) = List.map (fun p => p.1.length) (pairs t)"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data":
   "@weight_toWTree : ∀ {n : ℕ} (t : LTree n) (wt : Fin n → ℕ), weight (toWTree wt t) = (List.map wt (labels t)).sum"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 6},
   "data":
   "@wordsOf_forget : ∀ {n : ℕ} (t : LTree n), wordsOf t = leafWords (forget t)"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 6},
   "data":
   "'CompressionLean.huffman_optimal' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {

**Ce que l'annexe a exécuté** — l'arc Huffman complet du lake : la construction (`huffman`), sa lecture en coût de fusion (`costW`, `cost_dot`) et en code préfixe (`toCodeTree`, `trie_prefixFree`), la loi de la fusion (`subst`, `costW_subst`, `huffman_expand`), les mouvements locaux qui la forcent (`sisterPath`, `pathLen_le_maxDepth`, `both_mins_to_bottom`, `exchange_le`), et le verdict d'optimalité (`huffman_optimal`), sans un seul `sorry`. `SymbolCodec` referme la boucle avec le carnet 02 : le trie canonique sur symboles et l'arbre pondéré du lake sont le même objet, vu de deux côtés (`depthsW_toWTree`, `weight_toWTree`, `wordsOf_forget`).

In [17]:
-- Exercice 4 : la bande (1, 3, 3, 3, 5). Simulez les fusions a la main -- les
-- deux plus petits d'abord, payer leur somme (huffman_expand) -- et predisez
-- le cout total avant de lire la sortie. Indice : 1+3, puis 3+3, puis...
#eval costW (huffman [1, 3, 3, 3, 5])

-- Exercice 4 : la bande (1, 3, 3, 3, 5). Simulez les fusions a la main -- les
-- deux plus petits d'abord, payer leur somme (huffman_expand) -- et predisez
-- le cout total avant de lire la sortie. Indice : 1+3, puis 3+3, puis...
#eval costW (huffman [1, 3, 3, 3, 5])
─────▶  34
--% env 16

Raw input:
{"cmd": "-- Exercice 4 : la bande (1, 3, 3, 3, 5). Simulez les fusions a la main -- les\n-- deux plus petits d'abord, payer leur somme (huffman_expand) -- et predisez\n-- le cout total avant de lire la sortie. Indice : 1+3, puis 3+3, puis...\n#eval costW (huffman [1, 3, 3, 3, 5])", "env": 15}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "34"}],
 "env": 16}

## À retenir

Ce carnet a exécuté le lake, pas une copie : l'inégalité de Kraft (`kraft_le_one`), le round-trip de décodage (`decode_encode`) et la réciproque de McMillan (`mcmillan_converse`) sont les mêmes objets que ceux vérifiés par `lake build` et le gate `proof-integrity` en CI ; l'annexe 4 y ajoute l'arc Huffman jusqu'à `huffman_optimal`, compilé par le même `lake build`. Les ancres numériques rejoignent celles des carnets Python frères : le code (1, 2, 3, 3) du carnet 01/02 apparaît ici comme `sf4` (Kraft saturé), son message `a b c` se décode sans reste, `canon` le reconstruit mot pour mot depuis la seule bande de longueurs, et la bande de fréquences (4, 2, 1, 1) devient arbre de coût minimal (section 4).

**Suite de la série** — l'arc Huffman du lake (construction, coût de fusion, réarrangements, optimalité) est livré et exécuté en section 4 ; les plis à venir du lake s'écriront au même endroit.

## Vérifiez votre compréhension

1. Le code $(1, 2, 3, 3)$ des carnets 01/02 apparaît où dans le lake, et que dit `kraft_le_one` ?

<details><summary>Réponse</summary>

Comme `sf4` — un code préfixe de longueurs $(1,2,3,3)$ : la somme de Kraft
$\sum 2^{-\ell}$ vaut au plus 1 (`kraft_le_one`), ici exactement 1 : le
code est saturé (section 1).

</details>

2. Que garantit `decode_encode` sur le message `a b c` ?

<details><summary>Réponse</summary>

Le round-trip : décoder ce que l'encodage a produit redonne le message mot
pour mot, sans reste (section 2).

</details>

3. Que construit `mcmillan_converse` ?

<details><summary>Réponse</summary>

La réciproque de Kraft, **constructivement** : depuis une bande de
longueurs admissible, un code préfixe réalisé — l'exercice 3 en produit une
bande libre et la réalise (section 3).

</details>

4. Pourquoi l'arbre de `huffman ws` est-il de coût minimal ?

<details><summary>Réponse</summary>

`huffman_optimal` : pour tout arbre `t` de mêmes poids de feuilles (à
permutation près), `costW (huffman ws) ≤ costW t`. La preuve enchaîne la loi
de fusion (`huffman_expand` : chaque fusion paie exactement `w₁+w₂`) et les
réarrangements (`Appariement`, `Soeurs`, `Descente`, `DeuxMinima`,
`Echange`) qui montrent qu'un arbre mal apparié coûte plus cher
(section 4).

</details>

## Pour aller plus loin

- **Carnet suivant** : [Compression-02 — Huffman canonique](../Compression-02-HuffmanCanonique-Python.ipynb) — les
  longueurs deviennent table de décodage côté Python : ce que le lake
  prouve (`canon`) y tourne en bande passante réelle.
- **Carnet voisin** : [Compression-01 — ShannonFano](../Compression-01-ShannonFano-Prefixe-Python.ipynb), où le même
  code $(1,2,3,3)$ est construit à la main.
- **Le lake lui-même** : [README de `Compression_lean`](README.md) — les deux
  arcs du lake, Kraft–McMillan (sections 0-3) et Huffman (section 4), y sont
  documentés module par module.